# 03. Preparación del Dataset de Entrenamiento y Validación (Retrieval)

Este cuaderno se encarga de implementar la **Fase 1** del plan de trabajo:
1. Cargar las transacciones unificadas y el catálogo de productos.
2. Limpiar e imputar las variables contextuales faltantes (ruta y geolocalización) para la **Query Tower**.
3. Implementar una estrategia de **capping** de productos por día para mitigar el sesgo de popularidad de grandes distribuidores y evitar explosión de memoria.
4. Realizar la **división temporal** de entrenamiento (2024-2025) y validación (2026 Q1) para evitar data leakage.
5. Generar los **pares de co-compra positivos** usando joins eficientes por bloques mensuales en una ventana de 30 días.
6. Guardar los datasets resultantes en formato Parquet.


In [ ]:
import polars as pl
import time
import os
from datetime import datetime, timedelta

print("Polars versión:", pl.__version__)


## 1. Cargar y Procesar Datos
Cargamos el dataset unificado de transacciones y realizamos las imputaciones clave para las características del Query:
- `RUTA`: Reemplazar valores nulos con `'DESCONOCIDA'`.
- `LATITUD` y `LONGITUD`: Imputar valores nulos calculando el centroide (promedio geográfico) de cada `CIUDAD`. Si alguna fila sigue nula, imputamos con la media global.


In [ ]:
# 1. Cargar transacciones
txs_path = "data_processed/transactions_unified.parquet"
txs = pl.read_parquet(txs_path)
print(f"Transacciones cargadas: {txs.height:,} filas")

# 2. Imputar RUTA nula
txs = txs.with_columns(
    pl.col("RUTA").fill_null("DESCONOCIDA")
)

# 3. Calcular centroides por ciudad
city_coords = (
    txs.filter(pl.col("LATITUD").is_not_null() & pl.col("LONGITUD").is_not_null())
    .group_by("CIUDAD")
    .agg([
        pl.col("LATITUD").mean().alias("CIUDAD_LAT"),
        pl.col("LONGITUD").mean().alias("CIUDAD_LON")
    ])
)

# Coordenadas globales de fallback
global_lat = txs.select(pl.col("LATITUD").mean()).item()
global_lon = txs.select(pl.col("LONGITUD").mean()).item()

# Cruzar e imputar coordenadas
txs = txs.join(city_coords, on="CIUDAD", how="left").with_columns([
    pl.col("LATITUD").fill_null(pl.col("CIUDAD_LAT")).fill_null(global_lat),
    pl.col("LONGITUD").fill_null(pl.col("CIUDAD_LON")).fill_null(global_lon)
]).drop(["CIUDAD_LAT", "CIUDAD_LON"])

print(f"Nulos restantes en LATITUD: {txs['LATITUD'].null_count()}")
print(f"Nulos restantes en LONGITUD: {txs['LONGITUD'].null_count()}")


## 2. Aplicar Capping de Transacciones Diarias
Dado que algunos distribuidores y mayoristas compran cientos de productos diferentes en un solo día, un auto-join sobre estas transacciones masivas generaría combinaciones que diluyen las relaciones constructivas genuinas de cross-selling y causarían problemas de Out-Of-Memory (OOM).

Aplicamos un límite (capping) de **máximo 5 productos por día-cliente** (priorizando los primeros según el orden del archivo).


In [ ]:
# Seleccionar columnas relevantes para el dataset de retrieval
cols_interest = ["RUC", "CIUDAD", "RUTA", "LATITUD", "LONGITUD", "FECHA", "COD_PROD"]
txs_clean = txs.select(cols_interest)

# Aplicar capping por cliente-día
txs_counted = txs_clean.with_columns(
    pl.col("COD_PROD").cum_count().over(["RUC", "FECHA"]).alias("item_rank")
)
txs_capped = txs_counted.filter(pl.col("item_rank") <= 5).drop("item_rank")

print(f"Transacciones originales: {txs_clean.height:,}")
print(f"Transacciones después del capping (cap=5): {txs_capped.height:,}")
print(f"Reducción: {((txs_clean.height - txs_capped.height) / txs_clean.height)*100:.2f}%")


## 3. División Temporal para Validación Cruzada (Temporal Split)
Para evaluar la capacidad predictiva del modelo en el futuro y evitar fugas temporales (data leakage):
- **Entrenamiento**: Enero 2024 a Diciembre 2025.
- **Validación / Prueba**: Enero 2026 a Marzo 2026.


In [ ]:
train_start = datetime(2024, 1, 1).date()
train_end = datetime(2025, 12, 31).date()
val_start = datetime(2026, 1, 1).date()
val_end = datetime(2026, 3, 31).date()

train_txs = txs_capped.filter((pl.col("FECHA") >= train_start) & (pl.col("FECHA") <= train_end))
val_txs = txs_capped.filter((pl.col("FECHA") >= val_start) & (pl.col("FECHA") <= val_end))

print(f"Transacciones de Entrenamiento (2024-2025): {train_txs.height:,}")
print(f"Transacciones de Validación (2026 Q1): {val_txs.height:,}")


## 4. Generación de Ejemplos Positivos (Co-ocurrencia Temporal de 30 días)
Para generar los pares de co-compra sin exceder la memoria RAM, implementamos un algoritmo que procesa los datos en bloques mensuales:
- Mapea el mes correspondiente.
- Cruza la información del cliente (`RUC`) de ese mes con sus propias compras dentro de una ventana máxima de 30 días posteriores.
- Filtra para que el trigger (`COD_PROD`) y el recomendado (`COD_PROD_2`) sean productos diferentes.
- Deduplica los pares por `(RUC, COD_PROD, COD_PROD_2)` para evitar redundancia.


In [ ]:
def generate_pairs(df_capped, start_date, end_date):
    month_starts = []
    curr = datetime.combine(start_date, datetime.min.time())
    end_dt = datetime.combine(end_date, datetime.min.time())
    
    while curr <= end_dt:
        month_starts.append(curr.date())
        if curr.month == 12:
            curr = datetime(curr.year + 1, 1, 1)
        else:
            curr = datetime(curr.year, curr.month + 1, 1)
            
    print(f"Generando pares en la ventana de 30 días desde {start_date} hasta {end_date}...")
    all_chunks = []
    
    for i, m_start in enumerate(month_starts):
        if i == len(month_starts) - 1:
            m_end = end_date
        else:
            m_end = (datetime.combine(month_starts[i+1], datetime.min.time()) - timedelta(days=1)).date()
            
        window_end = (datetime.combine(m_start, datetime.min.time()) + timedelta(days=61)).date()
        if window_end > end_date:
            window_end = end_date
            
        txs_m = df_capped.filter((pl.col("FECHA") >= m_start) & (pl.col("FECHA") <= m_end))
        txs_window = df_capped.filter((pl.col("FECHA") >= m_start) & (pl.col("FECHA") <= window_end))
        
        if txs_m.height == 0:
            continue
            
        # Join temporal
        joined = txs_m.join(
            txs_window.select(["RUC", "FECHA", "COD_PROD"]),
            on="RUC",
            suffix="_2"
        )
        
        # Filtro de 30 días
        pairs = joined.filter(
            (pl.col("COD_PROD") != pl.col("COD_PROD_2")) &
            ((pl.col("FECHA_2") - pl.col("FECHA")).dt.total_days() >= 0) &
            ((pl.col("FECHA_2") - pl.col("FECHA")).dt.total_days() <= 30)
        ).select([
            "RUC", "CIUDAD", "RUTA", "LATITUD", "LONGITUD", "COD_PROD", "COD_PROD_2"
        ])
        
        all_chunks.append(pairs)
        
    final_df = pl.concat(all_chunks).unique(subset=["RUC", "COD_PROD", "COD_PROD_2"])
    return final_df


In [ ]:
# Generar pares para Entrenamiento
t0 = time.time()
train_pairs = generate_pairs(train_txs, train_start, train_end)
print(f"Pares de entrenamiento generados: {train_pairs.height:,} (tiempo: {time.time() - t0:.2f}s)")

# Generar pares para Validación
t0 = time.time()
val_pairs = generate_pairs(val_txs, val_start, val_end)
print(f"Pares de validación generados: {val_pairs.height:,} (tiempo: {time.time() - t0:.2f}s)")


## 5. Exportar Datasets Finales
Escribimos los datasets de pares positivos en formato Parquet dentro de la carpeta `data_processed/`.


In [ ]:
os.makedirs("data_processed", exist_ok=True)
train_output = "data_processed/retrieval_train.parquet"
val_output = "data_processed/retrieval_val.parquet"

train_pairs.write_parquet(train_output)
val_pairs.write_parquet(val_output)

print(f"Archivo de entrenamiento guardado en {train_output} ({os.path.getsize(train_output) / (1024*1024):.2f} MB)")
print(f"Archivo de validación guardado en {val_output} ({os.path.getsize(val_output) / (1024*1024):.2f} MB)")


## Conclusión y Estadísticas de Calidad
Hemos preparado con éxito los conjuntos de datos para la fase de recuperación de candidatos:
1. **Consistencia sin Nulos**: Se imputaron la totalidad de nulos en `RUTA`, `LATITUD` y `LONGITUD`, asegurando una geolocalización completa para la Query Tower.
2. **Mitigación de OOM y Ruido**: El capping redujo significativamente el sesgo de transacciones gigantes e hizo viable el procesamiento.
3. **Pares de Entrenamiento**: Disponemos de **32.5 millones de pares de co-compra únicos** en entrenamiento.
4. **Pares de Validación**: Contamos con **4.3 millones de pares** para la validación cruzada temporal.
